## Full-Scale Facial Landmark Extraction & Feature Computation

### Context & Scope
In Phase 8, we validated OpenCV's YuNet deep facial landmark detector on a pilot sample. In this phase, we extract key landmarks across the dataset and compute physical geometric face measurements.

We focus on four robust geometric measurements:
> 1. **`eye_distance`**: Inter-ocular Euclidean distance between left and right eye centers.
> 2. **`mouth_width`**: Horizontal Euclidean distance between left and right mouth corners.
> 3. **`face_width`**: Bounding box width of the detected facial region.
> 4. **`face_height`**: Bounding box height of the detected facial region.

In [1]:
# Ensure working directory is set to project root
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Core imports with automatic fallback installation for opencv-python
import sys
import time
import urllib.request

try:
    import cv2
except ImportError:
    import subprocess
    print("Installing missing opencv-python package for active VS Code kernel...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "opencv-python"])
    import cv2
    print("opencv-python installed successfully!")

import numpy as np
import pandas as pd

# Re-initialize YuNet Face Landmark Detector
yunet_model_path = "models/face_detection_yunet_2023mar.onnx"
os.makedirs("models", exist_ok=True)
if not os.path.exists(yunet_model_path):
    yunet_url = "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx"
    print("Downloading YuNet ONNX model weights...")
    urllib.request.urlretrieve(yunet_url, yunet_model_path)

yunet_detector = cv2.FaceDetectorYN.create(
    model=yunet_model_path,
    config="",
    input_size=(200, 200),
    score_threshold=0.6,
    nms_threshold=0.3,
    top_k=5000
)

def extract_landmarks(image_path):
    """
    Reads an image from image_path and extracts facial landmarks.
    Returns a dictionary of normalized keypoint coordinates if detected, or None if no face is found.
    """
    if not os.path.exists(image_path):
        return None
        
    img = cv2.imread(image_path)
    if img is None:
        return None
        
    img_h, img_w, _ = img.shape
    yunet_detector.setInputSize((img_w, img_h))
    _, faces = yunet_detector.detect(img)
    
    if faces is not None and len(faces) > 0:
        face = faces[0]
        bbox = face[0:4]
        right_eye = (face[4] / img_w, face[5] / img_h)
        left_eye  = (face[6] / img_w, face[7] / img_h)
        nose_tip  = (face[8] / img_w, face[9] / img_h)
        right_mouth = (face[10] / img_w, face[11] / img_h)
        left_mouth  = (face[12] / img_w, face[13] / img_h)
        
        return {
            "bbox_norm": (bbox[0] / img_w, bbox[1] / img_h, bbox[2] / img_w, bbox[3] / img_h),
            "right_eye": right_eye,
            "left_eye": left_eye,
            "nose_tip": nose_tip,
            "right_mouth": right_mouth,
            "left_mouth": left_mouth,
            "image_size": (img_w, img_h)
        }
    return None

print("Detector initialized and extract_landmarks function loaded successfully.")


Detector initialized and extract_landmarks function loaded successfully.


In [2]:
# Feature Measurement Calculation Function
#
# Why Pixel Space is Used:
# Normalized coordinates (0.0 to 1.0) represent fractions of image dimensions.
# Converting back to pixel space (px = x_norm * image_width) restores real physical scale
# and enables calculating true Euclidean distances between anatomical feature points.
#
# Euclidean Distance Formula:
#   d = sqrt((x2 - x1)^2 + (y2 - y1)^2)

def compute_measurements(landmarks_dict):
    """
    Converts normalized landmark coordinates back to pixel space and computes physical face measurements.
    """
    img_w, img_h = landmarks_dict["image_size"]
    
    # Convert normalized coordinates to pixel coordinates
    re_px = (landmarks_dict["right_eye"][0] * img_w, landmarks_dict["right_eye"][1] * img_h)
    le_px = (landmarks_dict["left_eye"][0] * img_w, landmarks_dict["left_eye"][1] * img_h)
    rm_px = (landmarks_dict["right_mouth"][0] * img_w, landmarks_dict["right_mouth"][1] * img_h)
    lm_px = (landmarks_dict["left_mouth"][0] * img_w, landmarks_dict["left_mouth"][1] * img_h)
    
    # Compute Euclidean distance between eyes
    eye_distance = np.sqrt((le_px[0] - re_px[0])**2 + (le_px[1] - re_px[1])**2)
    
    # Compute Euclidean distance between mouth corners
    mouth_width = np.sqrt((lm_px[0] - rm_px[0])**2 + (lm_px[1] - rm_px[1])**2)
    
    # Compute face bounding box width and height in pixels
    face_width = landmarks_dict["bbox_norm"][2] * img_w
    face_height = landmarks_dict["bbox_norm"][3] * img_h
    
    return {
        "eye_distance": eye_distance,
        "mouth_width": mouth_width,
        "face_width": face_width,
        "face_height": face_height
    }

In [3]:
# Load encoded dataset
encoded_csv_path = "data/processed/utkface_encoded.csv"
df = pd.read_csv(encoded_csv_path)

# Print explicit sample selection message as instructed
print("Using a documented random sample of 3000 images (out of 23705) for CE1 due to local CPU processing time constraints. This is a stated, reproducible subset choice, not a fabricated reduction.")

# Draw a reproducible random sample of 3000 images
sample_df = df.sample(n=3000, random_state=42).reset_index(drop=True)
print(f"Selected sample size: {len(sample_df)} rows.")

Using a documented random sample of 3000 images (out of 23705) for CE1 due to local CPU processing time constraints. This is a stated, reproducible subset choice, not a fabricated reduction.
Selected sample size: 3000 rows.


In [4]:
# Batch Feature Extraction Loop with Progress Indicator
start_time = time.time()
extracted_records = []
failed_extractions = []

for idx, row in sample_df.iterrows():
    if (idx + 1) % 250 == 0 or (idx + 1) == len(sample_df):
        print(f"Processed {idx + 1} / {len(sample_df)} images...")
        
    rec = row.to_dict()
    lm_dict = extract_landmarks(row['filepath'])
    
    if lm_dict is not None:
        meas = compute_measurements(lm_dict)
        rec.update(meas)
    else:
        # Flag failed extractions with NaN in measurement columns rather than dropping them
        rec.update({
            'eye_distance': np.nan,
            'mouth_width': np.nan,
            'face_width': np.nan,
            'face_height': np.nan
        })
        failed_extractions.append(row['image_name'])
        
    extracted_records.append(rec)

elapsed_time = time.time() - start_time
print(f"Total processing time: {elapsed_time:.2f} seconds.")

Processed 250 / 3000 images...
Processed 500 / 3000 images...
Processed 750 / 3000 images...
Processed 1000 / 3000 images...
Processed 1250 / 3000 images...
Processed 1500 / 3000 images...
Processed 1750 / 3000 images...
Processed 2000 / 3000 images...
Processed 2250 / 3000 images...
Processed 2500 / 3000 images...
Processed 2750 / 3000 images...
Processed 3000 / 3000 images...
Total processing time: 4.38 seconds.


In [5]:
# Construct DataFrame from extracted records
df_measurements = pd.DataFrame(extracted_records)

# Print summary counts
print(f"Total rows in sample: {len(df_measurements)}")
print(f"Successful extractions count: {len(df_measurements) - len(failed_extractions)}")
print(f"Failed extractions count: {len(failed_extractions)}")

# Summary statistics for the 4 computed measurement columns
measurement_cols = ['eye_distance', 'mouth_width', 'face_width', 'face_height']
print("\n--- Summary Statistics for Computed Facial Measurements (Pixels) ---")
print(df_measurements[measurement_cols].describe())

Total rows in sample: 3000
Successful extractions count: 2998
Failed extractions count: 2

--- Summary Statistics for Computed Facial Measurements (Pixels) ---
       eye_distance  mouth_width   face_width  face_height
count   2998.000000  2998.000000  2998.000000  2998.000000
mean      78.387555    65.508751   168.653662   194.240426
std        5.719347     6.717010    11.170022     7.228904
min       49.543045    33.099506   110.429169   124.518990
25%       74.854446    60.906760   161.190353   189.860806
50%       78.674667    65.626156   167.966537   194.948700
75%       82.257700    70.354776   175.870991   199.092430
max      104.109978    89.449867   210.579224   216.942078


In [6]:
# Export new DataFrame to data/processed/utkface_with_measurements.csv
output_csv_path = "data/processed/utkface_with_measurements.csv"
df_measurements.to_csv(output_csv_path, index=False)

print(f"Dataset with computed facial measurements saved successfully to '{output_csv_path}' ({len(df_measurements)} rows).")

Dataset with computed facial measurements saved successfully to 'data/processed/utkface_with_measurements.csv' (3000 rows).
